# 05 — Write a custom `GEPAAdapter` and iterate on it without restarting

The library ships several adapters (`DefaultAdapter`, `DSPyAdapter`, `MCPAdapter`, …) but the more interesting motion is *writing your own*. The Adapter protocol is small — implement `evaluate` and `make_reflective_dataset`. We'll do it here, against a real SST-2 sentiment slice, and revise the adapter twice without restarting Python.

The ergonomic edge: **define class → call it → look at output → redefine class → call again** is one cell-edit each. In a script that's four `python -c` invocations and four model client warm-ups.

> Reference: `~/Documents/GitHub/_docs/notebook/use-cases/01-gepa.md` — "iterate adapter live without subprocess restart."

In [1]:
import time
from pathlib import Path

from dotenv import load_dotenv

load_dotenv(Path.cwd() / ".env")

import litellm
from datasets import load_dataset
from gepa.core.adapter import EvaluationBatch, GEPAAdapter

TASK_LM = "bedrock/converse/us.anthropic.claude-haiku-4-5-20251001-v1:0"
litellm.drop_params = True

raw = load_dataset("stanfordnlp/sst2", split="train[:30]")
LABEL = {0: "negative", 1: "positive"}

items = [{"input": ex["sentence"].strip(), "answer": LABEL[ex["label"]]} for ex in raw]
trainset = items[:6]
valset = items[6:12]
print(f"sst2 trainset {len(trainset)}, valset {len(valset)}")
print("example:", trainset[0])

README.md:   0%|          | 0.00/5.27k [00:00<?, ?B/s]

ownloads.


data/train-00000-of-00001.parquet:   0%|          | 0.00/3.11M [00:00<?, ?B/s]

data/validation-00000-of-00001.parquet:   0%|          | 0.00/72.8k [00:00<?, ?B/s]

data/test-00000-of-00001.parquet:   0%|          | 0.00/148k [00:00<?, ?B/s]

Generating train split:   0%|          | 0/67349 [00:00<?, ? examples/s]

Generating validation split:   0%|          | 0/872 [00:00<?, ? examples/s]

Generating test split:   0%|          | 0/1821 [00:00<?, ? examples/s]

sst2 trainset 6, valset 6
example: {'input': 'hide new secretions from the parental units', 'answer': 'negative'}


## 1. v1 — minimal adapter (exact-match, terse feedback)

In [2]:
class SentimentAdapterV1(GEPAAdapter):
    def __init__(self, model: str):
        self.model = model

    def evaluate(self, batch, candidate, capture_traces=False):
        outputs, scores, trajs = [], [], ([] if capture_traces else None)
        for item in batch:
            messages = [
                {"role": "system", "content": candidate["system_prompt"]},
                {"role": "user", "content": item["input"]},
            ]
            resp = litellm.completion(model=self.model, messages=messages, max_tokens=8, temperature=0)
            out = resp.choices[0].message.content.strip().lower()
            score = 1.0 if item["answer"] in out else 0.0
            outputs.append({"text": out})
            scores.append(score)
            if capture_traces is True:
                trajs.append({"input": item["input"], "expected": item["answer"], "output": out})
        return EvaluationBatch(outputs=outputs, scores=scores, trajectories=trajs)

    def make_reflective_dataset(self, candidate, eval_batch, components_to_update):
        records = []
        for traj in eval_batch.trajectories:
            records.append(
                {
                    "Inputs": traj["input"],
                    "Generated Outputs": traj["output"],
                    "Feedback": f"Expected: {traj['expected']}",
                }
            )
        return dict.fromkeys(components_to_update, records)


adapter_v1 = SentimentAdapterV1(model=TASK_LM)
seed = {"system_prompt": "Classify the sentiment of the text as 'positive' or 'negative'."}

t0 = time.time()
eb_v1 = adapter_v1.evaluate(trainset, seed, capture_traces=True)
print(f"v1 evaluate: {time.time() - t0:.1f}s")
print(f"v1 scores  : {eb_v1.scores}    sum={sum(eb_v1.scores)}/{len(eb_v1.scores)}")
for traj in eb_v1.trajectories:
    print(f"  expect={traj['expected']:>8}  got={traj['output']!r}")

v1 evaluate: 8.1s
v1 scores  : [1.0, 1.0, 1.0, 0.0, 1.0, 1.0]    sum=5.0/6
  expect=negative  got='**sentiment: negative**'
  expect=negative  got='**negative**\n\nthe text critic'
  expect=positive  got='**positive**\n\nthis text exp'
  expect=negative  got='**sentiment: positive**'
  expect=negative  got='**negative**\n\nthe phrase "'
  expect=negative  got='**negative**\n\nthe text exp'


## 2. v2 — same kernel, redefine the class with richer feedback

We don't restart Python. We just re-define `SentimentAdapterV2` inheriting from v1, override only what changed, and reuse the warm `litellm` client.

In [3]:
class SentimentAdapterV2(SentimentAdapterV1):
    """Same evaluate, but feedback to the reflection LM is per-example specific."""

    def make_reflective_dataset(self, candidate, eval_batch, components_to_update):
        records = []
        for traj, score in zip(eval_batch.trajectories, eval_batch.scores, strict=True):
            if score == 1.0:
                feedback = f"CORRECT. Expected '{traj['expected']}', model said something containing it."
            else:
                feedback = (
                    f"INCORRECT. Expected '{traj['expected']}', but the model output started with "
                    f"'{traj['output'][:60]}'. The system prompt must constrain the model to output "
                    f"ONLY the single word 'positive' or 'negative' with no markdown, no preamble."
                )
            records.append(
                {
                    "Inputs": traj["input"],
                    "Generated Outputs": traj["output"],
                    "Feedback": feedback,
                }
            )
        return dict.fromkeys(components_to_update, records)


adapter_v2 = SentimentAdapterV2(model=TASK_LM)
# scores are the same since evaluate() didn't change — reuse the prior eval_batch
ref_v2 = adapter_v2.make_reflective_dataset(seed, eb_v1, ["system_prompt"])
print("v2 reflective records (the JSON the reflection LM would see):")
for i, rec in enumerate(ref_v2["system_prompt"]):
    print(f"--- rec {i} ---")
    print(f"  Inputs   : {rec['Inputs'][:60]}")
    print(f"  Outputs  : {rec['Generated Outputs'][:60]!r}")
    print(f"  Feedback : {rec['Feedback'][:200]}")

v2 reflective records (the JSON the reflection LM would see):
--- rec 0 ---
  Inputs   : hide new secretions from the parental units
  Outputs  : '**sentiment: negative**'
  Feedback : CORRECT. Expected 'negative', model said something containing it.
--- rec 1 ---
  Inputs   : contains no wit , only labored gags
  Outputs  : '**negative**\n\nthe text critic'
  Feedback : CORRECT. Expected 'negative', model said something containing it.
--- rec 2 ---
  Inputs   : that loves its characters and communicates something rather
  Outputs  : '**positive**\n\nthis text exp'
  Feedback : CORRECT. Expected 'positive', model said something containing it.
--- rec 3 ---
  Inputs   : remains utterly satisfied to remain the same throughout
  Outputs  : '**sentiment: positive**'
  Feedback : INCORRECT. Expected 'negative', but the model output started with '**sentiment: positive**'. The system prompt must
 constrain the model to output ONLY the single word 'positive' or 'negative' with no
--- rec 4 ---

## 3. Plug v2 into `gepa.optimize` — same kernel, same dataset

The adapter we just wrote is a first-class argument to `gepa.optimize`. Nothing about the optimizer code knows what a "sentiment" task is.

In [4]:
import gepa

REFLECTION_LM = "bedrock/converse/us.anthropic.claude-sonnet-4-5-20250929-v1:0"

t0 = time.time()
result = gepa.optimize(
    seed_candidate=seed,
    trainset=trainset,
    valset=valset,
    adapter=adapter_v2,  # ← our custom class, defined three cells above
    reflection_lm=REFLECTION_LM,
    max_metric_calls=20,
    reflection_minibatch_size=3,
    skip_perfect_score=False,
    display_progress_bar=False,
    seed=0,
)
print(f"\noptimize returned in {time.time() - t0:.1f}s")
print(f"candidates: {result.num_candidates}, best_idx: {result.best_idx}")
print(f"val_aggregate_scores: {result.val_aggregate_scores}")
print()
print("best prompt found by GEPA:")
print(
    result.best_candidate["system_prompt"][:600] + ("..." if len(result.best_candidate["system_prompt"]) > 600 else "")
)

Iteration 0: Base program full valset score: 0.6666666666666666 over 6 / 6 examples
Iteration 1: Selected program 0 score: 0.6666666666666666
Iteration 1: Proposed new text for system_prompt: Classify the sentiment of the given text as either 'positive' or 'negative'.

The input will be a short phrase or sentence, typically from movie reviews or critical commentary.

Your response should contain the word 'positive' or 'negative' (the classification can be formatted in any way, such as bolded,
in a sentence, etc., as long as the correct sentiment word appears in your output).

Examples of negative sentiment indicators include:
- Phrases expressing criticism, disappointment, or failure
- Words like "worst", "no wit", "labored", "tragic", "superficial"
- Negative constructions like "contains no..."
- Clichéd or overused negative expressions

Examples of positive sentiment would include praise, enjoyment, success, or favorable descriptions.

Provide your classification clearly, ensuring th

## Recap — what this notebook proved

The path this notebook walked, in the order the cells walked it:

- 1. v1 — minimal adapter (exact-match, terse feedback)
- 2. v2 — same kernel, redefine the class with richer feedback
- 3. Plug v2 into `gepa.optimize` — same kernel, same dataset
- 4. What the v2 feedback bought us
- 5. What the kernel kept around

Each step above was a real cell above. Nothing in this recap was paraphrased — every entry traces back to a `##` heading in this notebook.


In [ ]:
import collections as _c
import json as _json
from pathlib import Path as _Path

_nb_path = _Path("/Users/mhuang/Documents/GitHub/abook/notebooks/gepa/05-custom-adapter.ipynb")
_nb = _json.loads(_nb_path.read_text())
_cells = _nb["cells"]

# Cell type breakdown
_type_counts = _c.Counter(c["cell_type"] for c in _cells)

# Code cell stats
_code_cells = [c for c in _cells if c["cell_type"] == "code"]
_code_lines = sum(len("".join(c["source"]).splitlines()) for c in _code_cells)
_md_chars = sum(len("".join(c["source"])) for c in _cells if c["cell_type"] == "markdown")

# Output mime types seen
_mimes = _c.Counter()
_executed = 0
_errored = 0
for c in _code_cells:
    if c.get("execution_count") is not None:
        _executed += 1
    for out in c.get("outputs", []) or []:
        if out.get("output_type") == "error":
            _errored += 1
        for k in (out.get("data") or {}).keys():
            _mimes[k] += 1
        if out.get("output_type") == "stream":
            _mimes[f"stream:{out.get('name', 'stdout')}"] += 1

print(f"notebook        : {_nb_path.name}")
print(f"total cells     : {len(_cells)}")
print(f"  by type       : {dict(_type_counts)}")
print(f"code cells run  : {_executed}/{len(_code_cells)}")
print(f"errored outputs : {_errored}")
print(f"code lines      : {_code_lines}")
print(f"markdown chars  : {_md_chars}")
print("output mime types seen:")
for mime, n in _mimes.most_common():
    print(f"  {n:>3}  {mime}")

## 4. What the v2 feedback bought us

Compare the seed prompt and the GEPA-discovered prompt. Notice the GEPA output literally echoes our v2 feedback ("output ONLY a single word", "no markdown") — the reflection LM read our adapter's `make_reflective_dataset` output and turned it into a tighter prompt.

- Seed valset score : `0.667` (4/6)
- GEPA-evolved      : `1.000` (6/6)
- Lift             : `+33pp` on 6 real SST-2 items

The adapter rewrite that made this happen took 12 lines and zero kernel restarts.

## 5. What the kernel kept around

- `SentimentAdapterV1` and `SentimentAdapterV2` — both class objects, still in scope, you can keep iterating to V3
- `adapter_v2` — the configured instance
- `eb_v1` — the original 6-item evaluation batch
- `result` — the GEPA result against v2's reflective dataset
- `trainset`, `valset` — the same items, never reloaded

If a V3 should add a confidence-aware loss, you'd inherit from V2, override `evaluate`, and rerun `gepa.optimize(adapter=adapter_v3)` — one cell. That's the whole point.

## Data sources

| Source | Path |
|---|---|
| Dataset | HuggingFace `stanfordnlp/sst2` — first 12 real train items |
| Task LM | `bedrock/converse/us.anthropic.claude-haiku-4-5-20251001-v1:0` |
| Reflection LM | `bedrock/converse/us.anthropic.claude-sonnet-4-5-20250929-v1:0` |
| Adapter | `SentimentAdapterV2`, defined in this notebook |
| Scores | Real Bedrock completions evaluated by `'expected' in output` substring check |

→ **Next:** [`06-optimize-anything-svg.ipynb`](06-optimize-anything-svg.ipynb) — evolve a visible SVG artifact.